# 17 — Random Forest versus ResNet50
Use a **CPU runtime**, then Run all. This is a three-class real-data comparison with notebook 13's ResNet50, not the two-stage specialist from notebooks 15–16. Both use the same real training image pool and the same original and MILK validation partitions. The models use their own preprocessing; the CNN used transfer learning and standard image augmentation, while the forest uses fixed hand-engineered features. This compares complete approaches, not a controlled change to the classifier alone.

Use only image pixels for features. No dataset, demographic, clinical or file identifiers enter the forest. The forest configuration is fixed before evaluation. No test images are read, no deployment decision is made, and no app files change.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime,timezone
from PIL import Image,ImageOps
import hashlib,io,json,time,platform,os
import joblib
import sklearn
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd
from IPython.display import display

ROOT=Path('/content/drive/MyDrive/cnn-skin-cancer-classification')
SOURCE_RUN='exp08_572b2a3e5ca2'
SOURCE_MODELS=ROOT/'Models'/'Transfer_Learning'/SOURCE_RUN/'ResNet50'
SOURCE_REPORTS=ROOT/'Reports'/'Training'/SOURCE_RUN/'ResNet50'
MILK_ROOT=ROOT/'Datasets'/'MILK10k'/'Prepared'/'20261104_175741_606266'
F17_PATH=ROOT/'Reports'/'External_Data_Preparation'/'external_training_manifest.csv'
CLASS_NAMES=['BCC','Melanoma','SCC']
SEED=42
IMAGE_SIZE=(224,224)
RF_SIZE=(128,128)
RF_PARAMS=dict(n_estimators=300,max_depth=16,min_samples_leaf=3,
               max_features='sqrt',class_weight='balanced',random_state=SEED,n_jobs=2)

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
def write_json(path,value):
    path=Path(path);temporary=path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value,indent=2),encoding='utf-8');temporary.replace(path)
def validate_manifest(frame,split,name):
    required={'dataset','image_id','image_path','image_hash','class_label','label_index','split'}
    if not required.issubset(frame):raise ValueError(name+': missing columns')
    if frame.empty or not frame.split.eq(split).all():raise ValueError(name+': empty or wrong split')
    if not frame.class_label.map({c:i for i,c in enumerate(CLASS_NAMES)}).eq(frame.label_index).all():raise ValueError(name+': class encoding mismatch')
    if frame.duplicated(['dataset','image_id']).any() or frame.image_path.isna().any():raise ValueError(name+': duplicate IDs or missing paths')
    return frame.copy()

source_metadata=json.loads((SOURCE_MODELS/'metadata.json').read_text())
if not source_metadata.get('training_complete') or source_metadata['run_id']!=SOURCE_RUN:raise ValueError('Expected completed notebook 13 model.')
if source_metadata['class_names']!=CLASS_NAMES or source_metadata['preprocessing']!='resnet50' or source_metadata['image_size']!=list(IMAGE_SIZE):raise ValueError('Wrong source class order, preprocessing or shape.')
SOURCE_MODEL=SOURCE_MODELS/'best.keras'
if sha256_file(SOURCE_MODEL)!=source_metadata['model_sha256']:raise ValueError('Source model checksum mismatch.')
settings13=source_metadata['experiment_settings']
for split in ['train','validation','test']:
    if sha256_file(ROOT/'Datasets'/'Splits'/f'{split}_metadata.csv')!=settings13['original_split_sha256'][split]:raise ValueError('Original split changed: '+split)
    if sha256_file(MILK_ROOT/f'{split}_metadata.csv')!=settings13['milk_split_sha256'][split]:raise ValueError('MILK split changed: '+split)
if sha256_file(F17_PATH)!=settings13['external_training_manifest_sha256']:raise ValueError('Fitzpatrick training manifest changed.')
train_parts=[pd.read_csv(ROOT/'Datasets'/'Splits'/'train_metadata.csv'),pd.read_csv(F17_PATH),pd.read_csv(MILK_ROOT/'train_metadata.csv')]
full_train=validate_manifest(pd.concat(train_parts,ignore_index=True),'train','combined training')
train_frame=full_train
val_frames={'original':validate_manifest(pd.read_csv(ROOT/'Datasets'/'Splits'/'validation_metadata.csv'),'validation','original validation'),
            'MILK10k':validate_manifest(pd.read_csv(MILK_ROOT/'validation_metadata.csv'),'validation','MILK validation')}
fit_keys=set(zip(full_train.dataset,full_train.image_id))
for name,frame in val_frames.items():
    if set(zip(frame.dataset,frame.image_id))&fit_keys or set(frame.image_path)&set(full_train.image_path):raise ValueError('Training/validation overlap: '+name)
source_train=validate_manifest(pd.read_csv(SOURCE_REPORTS/'training_manifest.csv'),'train','source training')
cols=['dataset','image_id','class_label','image_hash','split']
a=full_train[cols].sort_values(['dataset','image_id']).reset_index(drop=True)
b=source_train[cols].sort_values(['dataset','image_id']).reset_index(drop=True)
pd.testing.assert_frame_equal(a,b,check_dtype=False)

settings={'source_run':SOURCE_RUN,'source_model_sha256':source_metadata['model_sha256'],
          'original_split_sha256':settings13['original_split_sha256'],
          'milk_split_sha256':settings13['milk_split_sha256'],
          'external_training_manifest_sha256':settings13['external_training_manifest_sha256'],
          'random_forest':RF_PARAMS,'feature_method':'rgb_hsv_histogram_uniform_lbp_glcm_v1',
          'feature_size':list(RF_SIZE),'sklearn_version':sklearn.__version__,
          'test_inference_performed':False,'synthetic_data':False}
stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
RUN_ID='rf_'+stamp
REPORT_ROOT=ROOT/'Reports'/'Random_Forest'/RUN_ID
MODEL_ROOT=ROOT/'Models'/'Random_Forest'/RUN_ID
for path in (REPORT_ROOT,MODEL_ROOT):path.mkdir(parents=True,exist_ok=False)
write_json(REPORT_ROOT/'settings.json',settings)
print('Run:',RUN_ID)
print('Real training images:',len(train_frame))
display(pd.crosstab(train_frame.dataset,train_frame.class_label))
print('Original validation:',len(val_frames['original']),'MILK validation:',len(val_frames['MILK10k']))
print('Test image bytes and predictions are not loaded.')

## Extract colour and texture features
Use global RGB and HSV histograms, a rotation-invariant uniform local binary pattern histogram and co-occurrence statistics at two pixel distances. The features can pick up lighting and background differences; performance must be reported separately by validation source.

In [ ]:
def reference_digest(raw,dataset):
    if dataset in {'Fitzpatrick17k','MILK10k'}:return hashlib.sha256(raw).hexdigest()
    if dataset not in {'HAM10000','PAD-UFES-20'}:raise ValueError('Unknown hash scheme: '+str(dataset))
    with Image.open(io.BytesIO(raw)) as image:
        rgb=ImageOps.exif_transpose(image).convert('RGB');h=hashlib.sha256()
        h.update(f'{rgb.width}x{rgb.height}:RGB:'.encode());h.update(rgb.tobytes())
    return h.hexdigest()

def extract_features(raw):
    with Image.open(io.BytesIO(raw)) as image:
        if image.width*image.height>20_000_000:raise ValueError('Image pixel limit exceeded.')
        image=image.convert('RGB').resize(RF_SIZE,Image.Resampling.BILINEAR)
        rgb=np.asarray(image)
        hsv=np.asarray(image.convert('HSV'))
        gray=np.asarray(image.convert('L'))
    features=[]
    for channels in (rgb,hsv):
        for channel in range(3):
            h=np.bincount((channels[:,:,channel]//16).ravel(),minlength=16)
            features.extend(h/h.sum())
    center=gray[1:-1,1:-1]
    neighbors=np.stack([gray[y:y+center.shape[0],x:x+center.shape[1]]>=center
                       for y,x in [(0,0),(0,1),(0,2),(1,2),(2,2),(2,1),(2,0),(1,0)]])
    transitions=(neighbors!=np.roll(neighbors,1,axis=0)).sum(axis=0)
    lbp=np.where(transitions<=2,neighbors.sum(axis=0),9)
    h=np.bincount(lbp.ravel(),minlength=10)
    features.extend(h/h.sum())
    bins=(gray//16).astype(np.int32)
    distance=np.abs(np.arange(16)[:,None]-np.arange(16)[None,:])
    for offset in (1,3):
        for a,b in [(bins[:,:-offset],bins[:,offset:]),(bins[:-offset,:],bins[offset:,:])]:
            matrix=np.bincount((16*a+b).ravel(),minlength=256).reshape(16,16)
            matrix=matrix+matrix.T
            matrix=matrix/matrix.sum()
            nonzero=matrix[matrix>0]
            features.extend([(matrix*distance**2).sum(),(matrix/(1+distance)).sum(),
                             (matrix**2).sum(),-(nonzero*np.log2(nonzero)).sum()])
    return np.asarray(features,dtype=np.float32)

def checked_bytes(row):
    path=Path(row['image_path'])
    raw=path.read_bytes()
    if reference_digest(raw,row['dataset'])!=str(row['image_hash']):
        raise ValueError('Source image checksum mismatch: '+str(path))
    return raw

def feature_matrix(frame,name):
    rows=[]
    for i,row in enumerate(frame.to_dict('records'),1):
        rows.append(extract_features(checked_bytes(row)))
        if i%500==0:print(name,i,'/',len(frame),flush=True)
    matrix=np.stack(rows)
    if not np.isfinite(matrix).all():raise ValueError('Invalid image features.')
    return matrix

started=time.perf_counter()
X_train=feature_matrix(train_frame,'Training')
X_val={name:feature_matrix(frame,name) for name,frame in val_frames.items()}
feature_seconds=time.perf_counter()-started
y_train=train_frame.label_index.to_numpy(dtype=np.int64)
if set(y_train)!={0,1,2}:raise ValueError('All three training classes required.')
forest=RandomForestClassifier(**RF_PARAMS)
started=time.perf_counter()
forest.fit(X_train,y_train)
fit_seconds=time.perf_counter()-started
np.testing.assert_array_equal(forest.classes_,[0,1,2])
forest.set_params(n_jobs=1)
FOREST_PATH=MODEL_ROOT/'random_forest.joblib'
joblib.dump(forest,FOREST_PATH,compress=3)
print('Features per image:',X_train.shape[1])
print('Feature extraction and fit seconds:',round(feature_seconds,2),round(fit_seconds,2))

In [ ]:
def class_metrics(labels,predicted):
    labels=np.asarray(labels,dtype=np.int64);predicted=np.asarray(predicted,dtype=np.int64)
    if labels.shape!=predicted.shape or labels.ndim!=1 or not len(labels):raise ValueError('Invalid classification labels.')
    if not np.isin(labels,[0,1,2]).all() or not np.isin(predicted,[0,1,2]).all():raise ValueError('Invalid class indices.')
    matrix=np.zeros((3,3),dtype=np.int64);np.add.at(matrix,(labels,predicted),1)
    support=matrix.sum(axis=1)
    if (support==0).any():raise ValueError('All three validation classes are required.')
    tp=matrix.diagonal().astype(float)
    precision=np.divide(tp,matrix.sum(axis=0),out=np.zeros(3),where=matrix.sum(axis=0)>0)
    recall=tp/support
    f1=np.divide(2*precision*recall,precision+recall,out=np.zeros(3),where=(precision+recall)>0)
    summary={'accuracy':float(tp.sum()/len(labels)),'macro_f1':float(f1.mean()),'BCC_recall':float(recall[0]),
             'Melanoma_recall':float(recall[1]),'SCC_precision':float(precision[2]),'SCC_recall':float(recall[2]),'SCC_f1':float(f1[2])}
    table=pd.DataFrame({'class_label':CLASS_NAMES,'precision':precision,'recall':recall,'f1':f1,'support':support,
                        'true_positive':tp.astype(int),'false_negative':support-tp.astype(int)})
    return summary,table,matrix


comparison=[];BASE_SCORES={};metric_results={}
for name,frame in val_frames.items():
    saved=pd.read_csv(SOURCE_REPORTS/f'best_{name}_validation_predictions.csv')
    if len(saved)!=len(frame) or not saved.split.eq('validation').all():
        raise ValueError('Wrong validation scores: '+name)
    aligned=frame[['dataset','image_id','class_label']].merge(saved,on=['dataset','image_id'],how='left',
              suffixes=('_expected',''),validate='one_to_one')
    if not aligned.class_label_expected.eq(aligned.class_label).all():
        raise ValueError('Saved CNN rows do not match the validation labels: '+name)
    base=aligned[[f'score_{c}' for c in CLASS_NAMES]].to_numpy(dtype=float)
    if not np.isfinite(base).all() or (base<0).any() or (base>1).any() or not np.allclose(base.sum(axis=1),1,atol=1e-4):
        raise ValueError('Invalid CNN scores: '+name)
    BASE_SCORES[name]=base
    labels=frame.label_index.to_numpy(dtype=np.int64)
    metric_results[name]={}
    for model_name,scores in [('ResNet50',base),('Random Forest',forest.predict_proba(X_val[name]))]:
        pred=scores.argmax(axis=1)
        summary,table,matrix=class_metrics(labels,pred)
        if model_name=='ResNet50':
            for key in ['accuracy','macro_f1','SCC_precision','SCC_recall','SCC_f1']:
                np.testing.assert_allclose(summary[key],source_metadata['validation_results_by_source'][name][key],rtol=0,atol=1e-7)
        metric_results[name][model_name]=summary
        comparison.append({'validation_set':name,'model':model_name,**summary})
        tag=model_name.lower().replace(' ','_')+'_'+name
        table.to_csv(REPORT_ROOT/f'{tag}_per_class.csv',index=False)
        pd.DataFrame(matrix,index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(REPORT_ROOT/f'{tag}_confusion.csv')
        out=frame[['dataset','image_id','class_label']].copy()
        out['predicted_class']=[CLASS_NAMES[i] for i in pred]
        for i,label in enumerate(CLASS_NAMES):out['score_'+label]=scores[:,i]
        out.to_csv(REPORT_ROOT/f'{tag}_predictions.csv',index=False)
comparison=pd.DataFrame(comparison)
comparison.to_csv(REPORT_ROOT/'validation_comparison.csv',index=False)
write_json(REPORT_ROOT/'validation_summary.json',metric_results)
display(comparison)
print('VALIDATION COMPARISON COMPLETE')
print('Reports:',REPORT_ROOT)

## Measure CPU latency
Compare one image at a time on the same CPU and the same 48 stratified validation images (8 per class per set). Time from image bytes in RAM to scores, including each model’s preprocessing. Verify the CNN’s scores against notebook 13 before timing. Exclude disk I/O, model loading and warm-up. These measurements describe this Colab runtime, not a clinic device. The validation report above is already saved if this timing cell fails.

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES']='-1'
os.environ['TF_NUM_INTRAOP_THREADS']='1'
os.environ['TF_NUM_INTEROP_THREADS']='1'
import tensorflow as tf
from threadpoolctl import threadpool_limits

tf.config.set_visible_devices([], 'GPU')
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
cnn=tf.keras.models.load_model(SOURCE_MODEL,compile=False,safe_mode=True)
def cnn_score(raw):
    image=tf.io.decode_image(raw,channels=3,expand_animations=False)
    image=tf.image.resize(tf.cast(image,tf.float32),IMAGE_SIZE,method='bilinear')
    image=tf.keras.applications.resnet50.preprocess_input(image)
    return cnn(image[None,...],training=False).numpy()[0]
def rf_score(raw):
    return forest.predict_proba(extract_features(raw)[None,:])[0]

sample=[]
for name,frame in val_frames.items():
    for label in CLASS_NAMES:
        selected=frame.index[frame.class_label.eq(label)]
        indices=np.random.default_rng(SEED).choice(selected,8,replace=False)
        for i in indices:
            row=frame.loc[i]
            sample.append((checked_bytes(row),BASE_SCORES[name][i]))
for raw,reference in sample:
    np.testing.assert_allclose(cnn_score(raw),reference,rtol=1e-3,atol=1e-4)
timings={'ResNet50':[],'Random Forest':[]}
predictors={'ResNet50':cnn_score,'Random Forest':rf_score}
with threadpool_limits(limits=1):
    for function in predictors.values():
        for raw,_ in sample[:3]:function(raw)
    for i,(raw,_) in enumerate(sample):
        order=list(predictors) if i%2==0 else list(reversed(predictors))
        for name in order:
            start=time.perf_counter()
            predictors[name](raw)
            timings[name].append((time.perf_counter()-start)*1000)
latency=pd.DataFrame([{'model':name,'images':len(values),'median_ms':float(np.median(values)),
                       'p95_ms':float(np.percentile(values,95))} for name,values in timings.items()])
latency.to_csv(REPORT_ROOT/'cpu_latency.csv',index=False)
write_json(REPORT_ROOT/'benchmark_context.json',{'platform':platform.platform(),
           'processor':platform.processor(),'logical_cpu_count':os.cpu_count(),
           'tensorflow_version':tf.__version__,'single_image':True,
           'cpu_threads_per_predictor':1,'includes_feature_extraction':True,
           'includes_disk_io_or_model_load':False,
           'random_forest_file_bytes':FOREST_PATH.stat().st_size,
           'resnet50_file_bytes':SOURCE_MODEL.stat().st_size})
display(comparison)
display(latency)
print('RANDOM FOREST COMPARISON COMPLETE')
print('Reports:',REPORT_ROOT)
print('No test evaluation or app replacement performed.')

## Save the results
Save the executed notebook as `notebooks/17_random_forest_baseline.ipynb`. Report SCC precision and recall, macro F1 and CPU timing for each approach. Keep the two validation sources separate and do not treat this comparison as clinical validation. Keep private models and prediction files in Drive.